In [ ]:
import os 
import csv
import json
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
label_counts = {
    "research_type": 0,
    # "result_outcome": 0,
    "affiliation": 0,
    # "problem_description": 0,
    # "goal_objective": 0,
    # "research_method": 0,
    # "research_question": 0,
    # "hypothesis": 0,
    # "prediction": 0,
    # "contribution": 0,
    "pseudocode": 0,
    "open_source_code": 0,
    # "open_experiment_code": 0,
    "train": 0,
    "validation": 0,
    # "test": 0,
    # "results": 0,
    "hardware_specification": 0,
    "software_dependencies": 0,
    "experiment_setup": 0,
}

In [ ]:
run_files = [
    "../../SOTA/results_run_01.csv",
    "../../SOTA/results_run_02.csv",
    "../../SOTA/results_run_03.csv",
    "../../SOTA/results_run_04.csv",
    "../../SOTA/results_run_05.csv",
]

In [ ]:
# Load manual evaluations
manual_evaluations_file = "../../evaluations.json"

with open(manual_evaluations_file, "r") as f:
    manual_evaluations = json.load(f)

In [ ]:
# Initialize a dictionary to store churn counts for each paper
paper_churn_counts = {}

# Iterate over each run file
for run_file in run_files:
    with open(run_file, "r") as f:
        reader = csv.DictReader(f)
        # Iterate over each row (paper) in the CSV
        for row in reader:
            # If this paper hasn't been seen before, initialize its label counts
            if row["filename"] not in paper_churn_counts:
                paper_churn_counts[row["filename"]] = label_counts.copy()

            # For each label, compare the run result with manual evaluation
            for label in label_counts.keys():
                # Normalize run label: treat 1 and 2 as 1 (positive)
                if int(row[label]) == 1 or int(row[label]) == 2:
                    row[label] = 1
                # Normalize manual evaluation: treat 1 and 2 as 1 (positive)
                if (
                    int(manual_evaluations[int(row["filename"]) - 1][label]) == 1
                    or int(manual_evaluations[int(row["filename"]) - 1][label]) == 2
                ):
                    manual_evaluations[int(row["filename"]) - 1][label] = 1

                # If run result matches manual evaluation, increment true count
                if (
                    int(row[label])
                    == manual_evaluations[int(row["filename"]) - 1][label]
                ):
                    paper_churn_counts[row["filename"]][label] += 1
    

In [ ]:
churns = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0}

label_churn_count = {
    "research_type": churns.copy(),
    # "result_outcome": churns.copy(),
    "affiliation": churns.copy(),
    # "problem_description": churns.copy(),
    # "goal_objective": churns.copy(),
    # "research_method": churns.copy(),
    # "research_question": churns.copy(),
    # "hypothesis": churns.copy(),
    # "prediction": churns.copy(),
    # "contribution": churns.copy(),
    "pseudocode": churns.copy(),
    "open_source_code": churns.copy(),
    # "open_experiment_code": churns.copy(),
    "train": churns.copy(),
    "validation": churns.copy(),
    # "test": churns.copy(),
    # "results": churns.copy(),
    "hardware_specification": churns.copy(),
    "software_dependencies": churns.copy(),
    "experiment_setup": churns.copy(),
}

In [ ]:
# Count the number of 5s in paper_churn_counts and update label_churn_count
for paper, counts in paper_churn_counts.items():
    for label, count in counts.items():
        if count >= 5:
            label_churn_count[label][5] += 1
        elif count >= 4:
            label_churn_count[label][4] += 1
        elif count >= 3:
            label_churn_count[label][3] += 1
        elif count >= 2:
            label_churn_count[label][2] += 1
        elif count >= 1:
            label_churn_count[label][1] += 1
        else:
            label_churn_count[label][0] += 1

print(label_churn_count)

In [ ]:
# calculate churn percentages
total_papers = len(paper_churn_counts)

churn_percentages = {
    label: {k: (v / total_papers) * 100 for k, v in counts.items()}
    for label, counts in label_churn_count.items()
}

# Print churn percentages as a dataframe
churn_df = pd.DataFrame(churn_percentages).T
churn_df.index.name = "Label"
churn_df.columns.name = "Churn Count"
print(churn_df)


In [ ]:
name_mapping = {
    "research_type": "Research Type",
    # "result_outcome": "Result Outcome",
    "affiliation": "Affiliation",
    # "problem_description": "Problem Description",
    # "goal_objective": "Goal/Objective",
    # "research_method": "Research Method",
    # "research_question": "Research Question",
    # "hypothesis": "Hypothesis",
    # "prediction": "Prediction",
    # "contribution": "Contribution",
    "pseudocode": "Pseudocode",
    "open_source_code": "Open Code",
    # "open_experiment_code": "Open Experiment Code",
    "train": "Open Datasets",
    "validation": "Dataset Splits",
    # "test": "Test",
    # "results": "Results",
    "hardware_specification": "Hardware Spec.",
    "software_dependencies": "Software Depend.",
    "experiment_setup": "Experiment Setup",
}

In [ ]:
font_size = 20

custom_colors = ["#f3b23e", "#e37137", "#d62728", "#e662bd", "#5dbaf7", "#9467bd"]

# Prepare data for plotting
labels = list(label_churn_count.keys())
x_labels = [name_mapping.get(label, label) for label in labels]
churn_levels = list(churns.keys())
churn_matrix = [
    [label_churn_count[label][level] for level in churn_levels] for label in labels
]

# Reverse churn_levels for stacking order
rev_churn_levels = churn_levels[::-1]

fig, ax = plt.subplots(figsize=(14, 7))
bottom = [0] * len(labels)
for idx, level in enumerate(rev_churn_levels):
    values = [label_churn_count[label][level] for label in labels]
    ax.bar(
        x_labels,
        values,
        bottom=bottom,
        label=f"{level} correct",
        color=custom_colors[idx],
    )
    bottom = [sum(x) for x in zip(bottom, values)]

plt.xticks(rotation=90)
plt.ylabel("Number of Papers", fontsize=font_size - 2)
# Show y-axis ticks and labels on both left and right
plt.gca().yaxis.set_ticks_position("both")
plt.gca().tick_params(axis="y", labelright=True)
plt.title("Reproducibility Variable Accuracy Count Distribution Over 5 Runs", fontsize=font_size)
plt.tight_layout()

# Start the y-axis at 200
plt.ylim(200, 405)

# Increase font size of tick labels
plt.xticks(fontsize=font_size - 4)
plt.yticks(fontsize=font_size - 4)

# Reduce the space between the bars and the y-axis
ax.margins(x=0.01)

# Reverse the legend order
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1], bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=font_size - 4)

# Rotate x-axis labels for better readability
plt.xticks(rotation=35, ha="right")

if not os.path.exists("../../plots"):
    os.makedirs("../../plots")

plt.savefig(f"../../plots/accuracy_consistency.pdf", dpi=600, bbox_inches='tight')

plt.show()